In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import dask
import zarr
import xarray as xr
import cftime
import os

In [4]:
# # ============== LOAD LLC ==============
# llc_path = '/orcd/data/abodner/002/cody/LLC_patch/LLC4320_face1_i2880-3600_j720-1440.zarr'
# llc_patch_full = xr.open_zarr(llc_path)

llc_patch_full = xr.open_zarr('/orcd/data/abodner/003/LLC4320/LLC4320',consolidated=False).isel(
    face=1,
    i=slice(2880,3600),
    i_g=slice(2880,3600),
    j=slice(720,1440),
    j_g=slice(720,1440),
)
# quad: i=[2880:4320), j=[0:1440)
# Agulhas: i=[2880:3600), j=[720:1440)
# ============== LOAD EMULATORS ==============
emulator_configs = [
    # {
    #     'name': 'rb-pred-resid-field-ckpt-50',
    #     'key': 'emulator_1',
    #     'path': '/orcd/data/abodner/002/cody/inference_patch/2026-07-20-eval:Samudra_LLC:rb-Agulhas-pred_field-eager-ckpt50-fixed-18406515/predictions_4d.zarr',
    #     'desc': ''
    # },
        {
        'name': 'rb-pred-resid-eager-ckpt-50',
        'key': 'emulator_1',
        'path': '/orcd/data/abodner/002/cody/inference_patch/2026-07-20-eval:Samudra_LLC:rb-Agulhas-pred_resid-eager-ckpt50-fixed-3weeks-18427424/predictions_4d_extended-18444486.zarr',
        'desc': ''
    },
    # {
    #     'name': 'rb-Agulhas-strides=1-pred_field-ckpt-25',
    #     'key': 'emulator_2',
    #     'path': '/orcd/data/abodner/002/cody/inference_patch/rb/2026-07-06-eval:Samudra_LLC:rb-Agulhas-strides=1-pred_field-ckpt-25-17335589/predictions_4d.zarr',
    #     'desc': ''   
    # },

]

# ============== OPEN EMULATOR DATASETS ==============
emulator_patches_raw = {}
for cfg in emulator_configs:
    emulator_patches_raw[cfg['key']] = xr.open_dataset(cfg['path'], consolidated=True)
    print(f"Loaded {cfg['name']}: {cfg['desc']}")

# ============== TIME MATCHING ==============
def normalize_times(times):
    return pd.DatetimeIndex([
        pd.Timestamp(
            int(t.year), int(t.month), int(t.day),
            int(t.hour), int(t.minute), int(t.second)
        )
        if hasattr(t, "year")
        else pd.Timestamp(t).floor("s")
        for t in times
    ])

llc_times_norm = normalize_times(llc_patch_full.time.values)

common_times = llc_times_norm
for cfg in emulator_configs:
    emulator_times_norm = normalize_times(emulator_patches_raw[cfg['key']].time.values)
    common_times = common_times.intersection(emulator_times_norm)

common_times = common_times.sort_values()

llc_mask = llc_times_norm.isin(common_times)
llc_patch = llc_patch_full.isel(time=llc_mask)

print(f"LLC subset to {len(common_times)} common times")

# ============== PORT GRID VARS & BUILD UNIFIED STRUCTURE ==============
grid_vars = ['XC', 'YC', 'rA', 'Z']

emulator_patches = {}
for cfg in emulator_configs:
    patch_raw = emulator_patches_raw[cfg['key']]
    patch_times_norm = normalize_times(patch_raw.time.values)

    patch_mask = patch_times_norm.isin(common_times)
    patch = patch_raw.isel(time=patch_mask)

    for gv in grid_vars:
        patch[gv] = llc_patch[gv]

    emulator_patches[cfg['key']] = patch

# ============== UNIFIED REFERENCE LISTS ==============
emulator_info = [(cfg['name'], cfg['key']) for cfg in emulator_configs]
n_emulators = len(emulator_info)

all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

print(f"\n=== Setup complete: LLC + {n_emulators} emulators ===")
for name, key in emulator_info:
    print(f"  {name} ({key})")

Loaded rb-pred-resid-eager-ckpt-50: 
LLC subset to 696 common times

=== Setup complete: LLC + 1 emulators ===
  rb-pred-resid-eager-ckpt-50 (emulator_1)


In [5]:
selected_time_range = [695, 696]   # inclusive indices
stepping = 1                    # 1 = every timestep, 4 = every 4th timestep

start_idx, end_idx = selected_time_range

# ----------------------------------------
# First subset LLC
# ----------------------------------------
llc_patch = llc_patch.isel(
    time=slice(start_idx, end_idx + 1, stepping)
)

# ----------------------------------------
# Then subset each emulator safely
# Handles shorter emulator runs automatically
# ----------------------------------------
emulator_patches_subset = {}

for key, patch in emulator_patches.items():

    max_time = patch.sizes['time']

    # Prevent indexing past emulator length
    safe_end_idx = min(end_idx, max_time - 1)

    patch_subset = patch.isel(
        time=slice(start_idx, safe_end_idx + 1, stepping)
    )

    emulator_patches_subset[key] = patch_subset

emulator_patches = emulator_patches_subset

# ----------------------------------------
# Match LLC length to shortest emulator
# ----------------------------------------
min_time_len = min(
    [llc_patch.sizes['time']] +
    [patch.sizes['time'] for patch in emulator_patches.values()]
)

llc_patch = llc_patch.isel(time=slice(0, min_time_len))

emulator_patches = {
    key: patch.isel(time=slice(0, min_time_len))
    for key, patch in emulator_patches.items()
}

# ----------------------------------------
# Rebuild combined dict
# ----------------------------------------
all_patches = {'llc': llc_patch}
all_patches.update(emulator_patches)

# ----------------------------------------
# Diagnostics
# ----------------------------------------
print(f"Subset to time indices {start_idx}:{end_idx}")
print(f"Stepping = {stepping}")
print(f"Final synchronized length = {min_time_len}")

print(f"LLC now has {llc_patch.sizes['time']} times")

for name, key in emulator_info:
    print(
        f"{name} ({key}) now has "
        f"{emulator_patches[key].sizes['time']} times"
    )

Subset to time indices 695:696
Stepping = 1
Final synchronized length = 1
LLC now has 1 times
rb-pred-resid-eager-ckpt-50 (emulator_1) now has 1 times


In [6]:
def format_time(t_val):
    """Format a time value to DD/MM/YYYY:HH regardless of cftime or datetime64."""
    try:
        return f"{t_val.day:02d}/{t_val.month:02d}/{t_val.year}:{t_val.hour:02d}h"
    except AttributeError:
        t_pd = pd.Timestamp(t_val)
        return f"{t_pd.day:02d}/{t_pd.month:02d}/{t_pd.year}:{t_pd.hour:02d}h"

# Gradients

In [7]:

# ============== INTEGRATED DIAGNOSTIC KNOBS ==============
grad_vars = ['Theta', 'Salt', 'U', 'V']
depth_vars = grad_vars.copy()
gradient_top_percentile = 98
single_time_plot = None  # Set to a time index, or None to skip the 1 x n single-time figures.

# ============== SURFACE GRADIENTS, NORMALIZED PER VARIABLE ==============
def _surface_gradient_magnitude(patch, var):
    """Surface horizontal gradient magnitude for one variable: (time, j, i)."""
    surface = patch[var].isel(k=0).values
    dx = np.sqrt(patch['rA'].values)
    dy = dx.copy()

    d_di = (
        np.roll(surface, -1, axis=2) - np.roll(surface, 1, axis=2)
    ) / (2 * dx[np.newaxis, :, :])
    d_dj = (
        np.roll(surface, -1, axis=1) - np.roll(surface, 1, axis=1)
    ) / (2 * dy[np.newaxis, :, :])

    return np.sqrt(d_di**2 + d_dj**2)


def _minmax_normalize(values, vmin, vmax):
    if not np.isfinite(vmin) or not np.isfinite(vmax) or np.isclose(vmax, vmin):
        return np.zeros_like(values, dtype=float)
    return (values - vmin) / (vmax - vmin)


surface_gradients_raw = {patch_name: {} for patch_name in all_patches}
surface_gradients_norm = {patch_name: {} for patch_name in all_patches}
gradient_norm_ranges = {}

for var in grad_vars:
    print(f"Computing surface gradient magnitudes for {var}...")
    var_min = np.inf
    var_max = -np.inf

    for patch_name, patch in all_patches.items():
        grad_mag = _surface_gradient_magnitude(patch, var)
        surface_gradients_raw[patch_name][var] = grad_mag

        if np.isfinite(grad_mag).any():
            var_min = min(var_min, np.nanmin(grad_mag))
            var_max = max(var_max, np.nanmax(grad_mag))

        print(f"  {patch_name}: {grad_mag.shape}")

    gradient_norm_ranges[var] = (var_min, var_max)
    for patch_name in all_patches:
        surface_gradients_norm[patch_name][var] = _minmax_normalize(
            surface_gradients_raw[patch_name][var], var_min, var_max
        )

    print(f"  normalized {var} with min={var_min:.6g}, max={var_max:.6g}")

# Mean of the four normalized variable gradients gives each variable equal weight.
integrated_surface_gradients = {}
gradient_masks = {}

for patch_name in all_patches:
    stacked = np.stack(
        [surface_gradients_norm[patch_name][var] for var in grad_vars],
        axis=0,
    )
    integrated_surface_gradients[patch_name] = np.nanmean(stacked, axis=0)

    n_times = integrated_surface_gradients[patch_name].shape[0]
    masks = np.zeros_like(integrated_surface_gradients[patch_name], dtype=bool)
    for t in range(n_times):
        field = integrated_surface_gradients[patch_name][t]
        threshold = np.nanpercentile(field, gradient_top_percentile)
        masks[t] = field >= threshold

    gradient_masks[patch_name] = masks
    print(
        f"Integrated {patch_name}: {masks.sum()} high-gradient surface pixels "
        f"({masks.sum() / masks.size * 100:.1f}%)"
    )

print("Done computing variable-integrated surface gradients!")


Computing surface gradient magnitudes for Theta...
  llc: (1, 720, 720)
  emulator_1: (1, 720, 720)
  normalized Theta with min=7.17916e-09, max=0.00558744
Computing surface gradient magnitudes for Salt...
  llc: (1, 720, 720)
  emulator_1: (1, 720, 720)
  normalized Salt with min=1.09584e-09, max=0.000671405
Computing surface gradient magnitudes for U...
  llc: (1, 720, 720)
  emulator_1: (1, 720, 720)
  normalized U with min=1.10451e-08, max=0.000670413
Computing surface gradient magnitudes for V...
  llc: (1, 720, 720)
  emulator_1: (1, 720, 720)
  normalized V with min=5.64461e-09, max=0.000552442
Integrated llc: 10368 high-gradient surface pixels (2.0%)
Integrated emulator_1: 10369 high-gradient surface pixels (2.0%)
Done computing variable-integrated surface gradients!


In [8]:

def _coord_block(patch, name, n_j, n_i):
    coord = patch[name]
    j_dim = 'j' if 'j' in coord.dims else ('lat' if 'lat' in coord.dims else 'y')
    i_dim = 'i' if 'i' in coord.dims else ('lon' if 'lon' in coord.dims else 'x')
    sel = coord.isel({
        j_dim: slice(0, n_j),
        i_dim: slice(0, n_i),
    })
    for extra_dim in set(sel.dims) - {j_dim, i_dim}:
        sel = sel.isel({extra_dim: 0})
    return sel.transpose(j_dim, i_dim).values


def _nice_lon_lat_ticks(coord_2d, axis_vals, axis):
    idx = np.linspace(0, len(axis_vals) - 1, 5).round().astype(int)
    pos = axis_vals[idx]
    if axis == 'i':
        coord_vals = coord_2d[coord_2d.shape[0] // 2, idx]
    else:
        coord_vals = coord_2d[idx, coord_2d.shape[1] // 2]
    return pos, [f'{v:.1f}' for v in coord_vals]


def _axes_grid(axes, nrows, ncols):
    return np.asarray(axes).reshape(nrows, ncols)


def _valid_time_indices(time_indices, n_times):
    return [t for t in time_indices if 0 <= t < n_times]


In [9]:

drift_colours = {
    'llc_only': '#FC9065',   # orange
    'emu_only': '#BE3977',   # pink
    'overlap':  '#FCFCBE',   # light tan
}


In [10]:
def plot_integrated_gradient_drift(time_indices, output_path, show_legend=True):
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = _valid_time_indices(time_indices, n_times)

    if not time_indices:
        print(f"No valid time indices for {output_path}; skipping.")
        return

    nrows = len(time_indices)
    ncols = n_emulators

    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.5*nrows), dpi=150)
    axes = _axes_grid(axes, nrows, ncols)

    sample_mask = gradient_masks['llc'][time_indices[0]]
    n_j, n_i = sample_mask.shape
    j_vals = np.arange(n_j)
    i_vals = np.arange(n_i)
    xc_block = _coord_block(llc_patch, 'XC', n_j, n_i)
    yc_block = _coord_block(llc_patch, 'YC', n_j, n_i)
    x_tick_pos, x_tick_labels = _nice_lon_lat_ticks(xc_block, i_vals, 'i')
    y_tick_pos, y_tick_labels = _nice_lon_lat_ticks(yc_block, j_vals, 'j')

    is_single_time_plot = len(time_indices) == 1

    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])
        llc_mask_surface = gradient_masks['llc'][t]
        target_pixels = np.sum(llc_mask_surface)

        for col, (emu_name, emu_key) in enumerate(emulator_info):
            ax = axes[row, col]

            theta_surface = all_patches[emu_key]['Theta'].isel(time=t, k=0).values
            ax.imshow(theta_surface, cmap='Greys', aspect='auto', origin='lower')

            emu_mask_surface = gradient_masks[emu_key][t]
            overlap_mask = llc_mask_surface & emu_mask_surface
            llc_only_mask = llc_mask_surface & ~emu_mask_surface
            emu_only_mask = emu_mask_surface & ~llc_mask_surface

            llc_j, llc_i = np.where(llc_only_mask)
            emu_j, emu_i = np.where(emu_only_mask)
            ovl_j, ovl_i = np.where(overlap_mask)

            ax.scatter(
                llc_i, llc_j,
                c=drift_colours['llc_only'],
                s=1,
                alpha=0.5,
                label='LLC integrated HG',
                rasterized=True,
            )
            ax.scatter(
                emu_i, emu_j,
                c=drift_colours['emu_only'],
                s=1,
                alpha=0.5,
                label='Emulator integrated HG',
                rasterized=True,
            )
            ax.scatter(
                ovl_i, ovl_j,
                c=drift_colours['overlap'],
                s=1,
                alpha=0.7,
                label='Overlap',
                rasterized=True,
            )

            n_overlap = np.sum(overlap_mask)
            overlap_pct = round(100 * n_overlap / target_pixels, 2)

            if is_single_time_plot:
                title = (
                    f'step={int(time_indices[0])}, '
                    f'overlap={n_overlap}/{target_pixels}, {overlap_pct}%'
                )
            else:
                title = f'overlap={n_overlap}/{target_pixels}, {overlap_pct}%'

            ax.set_title(title, fontsize=8)

            ax.set_xticks(x_tick_pos)
            ax.set_xticklabels(x_tick_labels, fontsize=7)
            ax.set_yticks(y_tick_pos)
            ax.set_yticklabels(y_tick_labels, fontsize=7)
            ax.set_xlabel('Longitude', fontsize=7)
            ax.set_ylabel('Latitude', fontsize=7)

            if show_legend and row == 0 and col == ncols - 1:
                ax.legend(fontsize=5, loc='upper right', markerscale=5)

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved {output_path}")


os.makedirs('figs/gradients/variable_integrated', exist_ok=True)
ref_patch = emulator_patches[emulator_info[0][1]]
all_time_indices = list(range(len(ref_patch.time)))

plot_integrated_gradient_drift(
    all_time_indices,
    'figs/gradients/variable_integrated/surface_gradient_drift_all_times.png',
)

if single_time_plot is not None:
    plot_integrated_gradient_drift(
        [int(single_time_plot)],
        f'figs/gradients/variable_integrated/surface_gradient_drift_time_{int(single_time_plot):03d}.png',
        show_legend=False,
    )

print("Done with variable-integrated surface gradient drift figures!")

Saved figs/gradients/variable_integrated/surface_gradient_drift_all_times.png
Done with variable-integrated surface gradient drift figures!


# Error vs depth plots

In [11]:
error_colours = {
    'mean': '#631980',
    'median': '#BE3977',
    'hg_mean': '#FC9065',
    'std': 'k',
}


def compute_raw_error_stats_for_var(emu_key, var):
    """Compute raw per-depth absolute-error statistics for one variable."""
    n_times = llc_patch.sizes['time']
    n_depths = llc_patch.sizes['k']

    var_stats = {
        'mean': np.full((n_times, n_depths), np.nan),
        'median': np.full((n_times, n_depths), np.nan),
        'std': np.full((n_times, n_depths), np.nan),
        'hg_mean': np.full((n_times, n_depths), np.nan),
    }

    print(f"  computing raw {emu_key} {var} error stats...")

    for t in range(n_times):
        hg_mask = gradient_masks['llc'][t]

        llc_data = llc_patch[var].isel(time=t).values
        emu_data = emulator_patches[emu_key][var].isel(time=t).values
        abs_error = np.abs(llc_data - emu_data)

        for k in range(n_depths):
            depth_values = abs_error[k].reshape(-1)
            hg_values = abs_error[k][hg_mask].reshape(-1)

            var_stats['mean'][t, k] = np.nanmean(depth_values)
            var_stats['median'][t, k] = np.nanmedian(depth_values)
            var_stats['std'][t, k] = np.nanstd(depth_values)
            var_stats['hg_mean'][t, k] = (
                np.nanmean(hg_values) if hg_values.size else np.nan
            )

    return var_stats


raw_error_stats = {}

for emu_name, emu_key in emulator_info:
    raw_error_stats[emu_key] = {}

    for var in depth_vars:
        print(f"Computing raw error stats for {emu_name}, {var}...")
        raw_error_stats[emu_key][var] = compute_raw_error_stats_for_var(
            emu_key,
            var,
        )

print("Done computing raw error stats!")


def _raw_error_stats_for_var(emu_key, var, time_indices):
    """Return raw stats for one variable and selected times."""
    var_stats = raw_error_stats[emu_key][var]

    return {
        'mean': var_stats['mean'][time_indices],
        'median': var_stats['median'][time_indices],
        'std': var_stats['std'][time_indices],
        'hg_mean': var_stats['hg_mean'][time_indices],
    }


def plot_error_stats_for_var(var, time_indices, output_path):
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = _valid_time_indices(time_indices, n_times)

    if not time_indices:
        print(f"No valid time indices for {output_path}; skipping.")
        return

    nrows = len(time_indices)
    ncols = n_emulators

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(4*ncols, 3*nrows),
        dpi=150,
    )
    axes = _axes_grid(axes, nrows, ncols)

    depths = -1 * np.round(llc_patch.Z.values).astype(int)

    stats_by_emulator = {}

    for _, emu_key in emulator_info:
        stats_by_emulator[emu_key] = _raw_error_stats_for_var(
            emu_key,
            var,
            time_indices,
        )

    global_xmax = 0.0
    for emu_key in stats_by_emulator:
        stats = stats_by_emulator[emu_key]

        this_xmax = np.nanmax([
            np.nanmax(stats['median']),
            np.nanmax(stats['mean']),
            np.nanmax(stats['std']),
            np.nanmax(stats['hg_mean']),
        ])

        if np.isfinite(this_xmax):
            global_xmax = max(global_xmax, this_xmax)

    if not np.isfinite(global_xmax) or global_xmax <= 0:
        global_xmax = 1.0

    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])

        for col, (emu_name, emu_key) in enumerate(emulator_info):
            ax = axes[row, col]
            stats = stats_by_emulator[emu_key]

            ax.scatter(
                stats['median'][row],
                depths,
                color=error_colours['median'],
                s=30,
                alpha=0.7,
                zorder=3,
            )
            ax.plot(
                stats['median'][row],
                depths,
                color=error_colours['median'],
                alpha=0.4,
                linewidth=1.5,
                label='Median',
            )

            ax.scatter(
                stats['mean'][row],
                depths,
                color=error_colours['mean'],
                s=30,
                alpha=0.7,
                zorder=4,
            )
            ax.plot(
                stats['mean'][row],
                depths,
                color=error_colours['mean'],
                alpha=0.4,
                linewidth=1.5,
                label='Mean',
            )

            ax.scatter(
                stats['std'][row],
                depths,
                color=error_colours['std'],
                s=30,
                alpha=0.7,
                zorder=5,
            )
            ax.plot(
                stats['std'][row],
                depths,
                color=error_colours['std'],
                alpha=0.4,
                linewidth=1.5,
                label='Std',
            )

            ax.scatter(
                stats['hg_mean'][row],
                depths,
                color=error_colours['hg_mean'],
                s=30,
                alpha=0.7,
                zorder=6,
            )
            ax.plot(
                stats['hg_mean'][row],
                depths,
                color=error_colours['hg_mean'],
                alpha=0.4,
                linewidth=1.5,
                label='HG Mean',
            )

            ax.set_title(
                f'{var}',#f'{emu_name} {var} error {time_str}',
                fontsize=12,
            )
            ax.set_xlabel('Absolute Error', fontsize=7)

            if col == 0:
                ax.set_ylabel('Depth (m)', fontsize=7)
            else:
                ax.set_ylabel('')
                ax.tick_params(axis='y', left=False, labelleft=False)

            ax.set_ylim(1000, 0)
            ax.set_yticks([0, 250, 500, 750, 1000])
            ax.set_xlim(0, global_xmax * 1.05)

            ax.grid(alpha=0.2)
            ax.tick_params(labelsize=6)
            ax.set_facecolor('#E3E3E3')

            # if col == 0:
            #     ax.legend(fontsize=6, loc='lower right')

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved {output_path}")


os.makedirs('figs/prognostic_var_comparison/variable_integrated', exist_ok=True)

ref_patch = emulator_patches[emulator_info[0][1]]
all_time_indices = list(range(len(ref_patch.time)))

for var in depth_vars:
    plot_error_stats_for_var(
        var,
        all_time_indices,
        f'figs/prognostic_var_comparison/variable_integrated/depth_error_stats_{var}.png',
    )

if single_time_plot is not None:
    for var in depth_vars:
        plot_error_stats_for_var(
            var,
            [int(single_time_plot)],
            f'figs/prognostic_var_comparison/variable_integrated/depth_error_stats_{var}_time_{int(single_time_plot):03d}.png',
        )

#print("Done with variable-separated raw error vs depth figures!")

Computing raw error stats for rb-pred-resid-eager-ckpt-50, Theta...
  computing raw emulator_1 Theta error stats...
Computing raw error stats for rb-pred-resid-eager-ckpt-50, Salt...
  computing raw emulator_1 Salt error stats...
Computing raw error stats for rb-pred-resid-eager-ckpt-50, U...
  computing raw emulator_1 U error stats...
Computing raw error stats for rb-pred-resid-eager-ckpt-50, V...
  computing raw emulator_1 V error stats...
Done computing raw error stats!
Saved figs/prognostic_var_comparison/variable_integrated/depth_error_stats_Theta.png
Saved figs/prognostic_var_comparison/variable_integrated/depth_error_stats_Salt.png
Saved figs/prognostic_var_comparison/variable_integrated/depth_error_stats_U.png
Saved figs/prognostic_var_comparison/variable_integrated/depth_error_stats_V.png


In [55]:

error_colours = {
    'mean': '#631980',
    'median': '#BE3977',
    'hg_mean': '#FC9065',
    'std': 'k',
}


def _safe_nanminmax(values):
    if not np.isfinite(values).any():
        return np.nan, np.nan
    return np.nanmin(values), np.nanmax(values)


def _normalise_stat_values(values, vmin, vmax):
    if not np.isfinite(vmin) or not np.isfinite(vmax) or np.isclose(vmax, vmin):
        return np.zeros_like(values, dtype=float)
    return (values - vmin) / (vmax - vmin)


def _stat_norm_range(var_stats, time_indices):
    pieces = []
    for stat_name in ['mean', 'median', 'std', 'hg_mean']:
        pieces.append(var_stats[stat_name][time_indices].reshape(-1))
    return _safe_nanminmax(np.concatenate(pieces))


def compute_raw_error_stats(emu_key):
    """Compute per-variable depth statistics before any normalization."""
    n_times = llc_patch.sizes['time']
    n_depths = llc_patch.sizes['k']
    emu_stats = {}

    for var in depth_vars:
        print(f"  computing raw {emu_key} {var} error stats...")
        var_stats = {
            'mean': np.full((n_times, n_depths), np.nan),
            'median': np.full((n_times, n_depths), np.nan),
            'std': np.full((n_times, n_depths), np.nan),
            'hg_mean': np.full((n_times, n_depths), np.nan),
        }

        for t in range(n_times):
            hg_mask = gradient_masks['llc'][t]
            llc_data = llc_patch[var].isel(time=t).values
            emu_data = emulator_patches[emu_key][var].isel(time=t).values
            abs_error = np.abs(llc_data - emu_data)

            for k in range(n_depths):
                depth_values = abs_error[k].reshape(-1)
                hg_values = abs_error[k][hg_mask].reshape(-1)

                var_stats['mean'][t, k] = np.nanmean(depth_values)
                var_stats['median'][t, k] = np.nanmedian(depth_values)
                var_stats['std'][t, k] = np.nanstd(depth_values)
                var_stats['hg_mean'][t, k] = np.nanmean(hg_values) if hg_values.size else np.nan

        emu_stats[var] = var_stats

    return emu_stats


raw_error_stats = {}
for emu_name, emu_key in emulator_info:
    print(f"Computing raw variable-wise error stats for {emu_name}...")
    raw_error_stats[emu_key] = compute_raw_error_stats(emu_key)

print("Done computing raw error stats!")


def _integrated_normalized_error_stats(emu_key, time_indices):
    """Normalize each variable's stats after stat calculation, then average vars."""
    n_depths = llc_patch.sizes['k']
    integrated = {
        'mean': np.zeros((len(time_indices), n_depths), dtype=float),
        'median': np.zeros((len(time_indices), n_depths), dtype=float),
        'std': np.zeros((len(time_indices), n_depths), dtype=float),
        'hg_mean': np.zeros((len(time_indices), n_depths), dtype=float),
    }
    norm_ranges = {}

    for var in depth_vars:
        var_stats = raw_error_stats[emu_key][var]
        vmin, vmax = _stat_norm_range(var_stats, time_indices)
        norm_ranges[var] = (vmin, vmax)

        for stat_name in integrated:
            normalized = _normalise_stat_values(
                var_stats[stat_name][time_indices],
                vmin,
                vmax,
            )
            integrated[stat_name] += normalized / len(depth_vars)

    return integrated, norm_ranges


def plot_integrated_error_stats(time_indices, output_path):
    ref_patch = emulator_patches[emulator_info[0][1]]
    n_times = len(ref_patch.time)
    time_indices = _valid_time_indices(time_indices, n_times)

    if not time_indices:
        print(f"No valid time indices for {output_path}; skipping.")
        return

    nrows = len(time_indices)
    ncols = n_emulators
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3*nrows), dpi=150)
    axes = _axes_grid(axes, nrows, ncols)

    depths = -1 * np.round(llc_patch.Z.values).astype(int)

    normalized_stats_by_emulator = {}
    norm_ranges_by_emulator = {}
    for _, emu_key in emulator_info:
        normalized_stats_by_emulator[emu_key], norm_ranges_by_emulator[emu_key] = _integrated_normalized_error_stats(
            emu_key,
            time_indices,
        )

    for row, t in enumerate(time_indices):
        time_str = format_time(ref_patch.time.values[t])

        for col, (emu_name, emu_key) in enumerate(emulator_info):
            ax = axes[row, col]
            stats = normalized_stats_by_emulator[emu_key]

            ax.scatter(stats['median'][row], depths, color=error_colours['median'], s=30, alpha=0.7, zorder=3)
            ax.plot(stats['median'][row], depths, color=error_colours['median'], alpha=0.4, linewidth=1.5, label='Median')

            ax.scatter(stats['mean'][row], depths, color=error_colours['mean'], s=30, alpha=0.7, zorder=4)
            ax.plot(stats['mean'][row], depths, color=error_colours['mean'], alpha=0.4, linewidth=1.5, label='Mean')

            ax.scatter(stats['std'][row], depths, color=error_colours['std'], s=30, alpha=0.7, zorder=5)
            ax.plot(stats['std'][row], depths, color=error_colours['std'], alpha=0.4, linewidth=1.5, label='Std')

            ax.scatter(stats['hg_mean'][row], depths, color=error_colours['hg_mean'], s=30, alpha=0.7, zorder=5)
            ax.plot(stats['hg_mean'][row], depths, color=error_colours['hg_mean'], alpha=0.4, linewidth=1.5, label='HG Mean')



            if row == 0:
                ax.set_title(
                    var,
                    fontsize=12,
                    fontweight='bold',
                )
            ax.set_xlabel('Post-stat Normalized Error', fontsize=7)
            ax.set_ylabel('Depth (m)', fontsize=7)
            ax.set_ylim(1000, 0)
            ax.set_yticks([0, 250, 500, 750, 1000])
            xmax = np.nanmax([
                np.nanmax(stats['median'][row]),
                np.nanmax(stats['mean'][row] + stats['std'][row]),
                np.nanmax(stats['hg_mean'][row]),
            ])
            ax.set_xlim(0, max(1.05, xmax * 1.05))
            ax.grid(alpha=0.2)
            ax.tick_params(labelsize=6)
            ax.set_facecolor('#E3E3E3')

            if col == 0:
                ax.legend(fontsize=6, loc='lower right')

    plt.tight_layout()
    plt.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved {output_path}")


os.makedirs('figs/prognostic_var_comparison/variable_integrated', exist_ok=True)
ref_patch = emulator_patches[emulator_info[0][1]]
all_time_indices = list(range(len(ref_patch.time)))

plot_integrated_error_stats(
    all_time_indices,
    'figs/prognostic_var_comparison/variable_integrated/depth_error_stats_all_times.png',
)

if single_time_plot is not None:
    plot_integrated_error_stats(
        [int(single_time_plot)],
        f'figs/prognostic_var_comparison/variable_integrated/depth_error_stats_time_{int(single_time_plot):03d}.png',
    )

print("Done with post-stat normalized variable-integrated error vs depth figures!")


Computing raw variable-wise error stats for exp3_ckpt38...
  computing raw emulator_1 Theta error stats...
  computing raw emulator_1 Salt error stats...
  computing raw emulator_1 U error stats...
  computing raw emulator_1 V error stats...
Done computing raw error stats!
Saved figs/prognostic_var_comparison/variable_integrated/depth_error_stats_all_times.png
Saved figs/prognostic_var_comparison/variable_integrated/depth_error_stats_time_018.png
Done with post-stat normalized variable-integrated error vs depth figures!
